# Exploración de intervenciones con Pandas (laboratorio L4)

**Guías:** Python semana 2 (estructuras, NumPy y Pandas) y Python semana 5 (transformaciones con Pandas y NumPy, trazabilidad).

**Objetivo.** Explorar las intervenciones de mantenimiento biomédico de SITBIO y preparar un conjunto derivado y
trazable para la analítica de HU-008 (indicadores) y HU-009 (agrupación de equipos), sin modificar la fuente.

**Fuente.** `seeds/mongo/data/intervenciones.ejson.json`: 12 intervenciones **100 % ficticias** de los dos tenants de
laboratorio (Clínica Demo A y Clínica Demo B), en Extended JSON. Se leen con `bson.json_util.loads`, así que no hace
falta un servidor MongoDB. Las ubicaciones actuales de los equipos salen de `seeds/mongo/data/referencias_postgres.json`.

**Reglas del notebook.** Rutas relativas con `pathlib`; fechas en UTC; la fuente no se modifica (se trabaja sobre una
copia); ninguna fila se borra (los problemas se **marcan**); cuando falta un dato no se inventa un valor.

## 0. Entorno y rutas

In [ ]:
import hashlib
import os
import platform
from pathlib import Path

import numpy as np
import pandas as pd
from bson import json_util

print(f"Python {platform.python_version()} · Pandas {pd.__version__} · NumPy {np.__version__}")


def raiz_repositorio(inicio: Path) -> Path:
    """Sube carpetas hasta encontrar pyproject.toml (la raíz del repositorio)."""
    for carpeta in [inicio, *inicio.parents]:
        if (carpeta / "pyproject.toml").exists():
            return carpeta
    raise FileNotFoundError("No se encontró la raíz del repositorio (pyproject.toml).")


RAIZ = raiz_repositorio(Path.cwd().resolve())
FUENTE = RAIZ / "seeds" / "mongo" / "data" / "intervenciones.ejson.json"
REFERENCIAS = RAIZ / "seeds" / "mongo" / "data" / "referencias_postgres.json"
SALIDA = RAIZ / "analytics" / "data" / "processed"
SALIDA.mkdir(parents=True, exist_ok=True)

TENANTS = {
    "11111111-1111-4111-8111-111111111111": "Clínica Demo A",
    "22222222-2222-4222-8222-222222222222": "Clínica Demo B",
}
print("Fuente:", FUENTE.relative_to(RAIZ).as_posix())

## 1. Trazabilidad: huella SHA-256 de la fuente

La huella (hash SHA-256) identifica **exactamente** el archivo que se analizó: si cambia un solo byte, cambia la
huella. Se calcula al inicio y se vuelve a calcular al final para demostrar que la fuente quedó intacta; también se
escribe en la bitácora de reglas, para que cualquiera pueda comprobar con qué datos se produjeron los resultados.

In [ ]:
def sha256(ruta: Path) -> str:
    return hashlib.sha256(ruta.read_bytes()).hexdigest()


HUELLA_FUENTE = sha256(FUENTE)
print("SHA-256 de la fuente:", HUELLA_FUENTE)

## 2. Estructuras básicas (semana 2)

`json_util.loads` convierte el Extended JSON en tipos de Python: `{"$date": ...}` pasa a `datetime` y
`{"$numberDouble": ...}` a `float`. El resultado es una **lista de diccionarios**, la misma forma que devolvería
PyMongo con `list(coleccion.find(...))`.

In [ ]:
intervenciones = json_util.loads(FUENTE.read_text(encoding="utf-8"))
print(type(intervenciones).__name__, len(intervenciones), "documentos")
primera = intervenciones[0]
tipos = {k: type(v).__name__ for k, v in primera.items()}
print({k: tipos[k] for k in ("fecha_intervencion", "costo_total", "repuestos")})

**Celda opcional (servidor MongoDB).** Si la variable de entorno `SITBIO_MONGO_URI` existe, se cuentan los documentos
de la colección con PyMongo para compararlos con el archivo. Sin la variable, la celda no hace nada; el notebook
siempre analiza el archivo de la semilla, que es reproducible.

In [ ]:
uri = os.environ.get("SITBIO_MONGO_URI", "").strip()
if uri:
    from pymongo import MongoClient

    with MongoClient(uri, serverSelectionTimeoutMS=3000) as cliente:
        base = cliente[os.environ.get("SITBIO_MONGO_DB", "sitbio_dev")]
        print("Documentos en MongoDB:", base["intervenciones"].count_documents({}))
else:
    print("SITBIO_MONGO_URI no está definida: se usa solo el archivo de la semilla.")

**NumPy sobre `costo_total`.** Un arreglo NumPy guarda números del mismo tipo en memoria contigua y opera sobre todos
a la vez (vectorización), sin ciclos `for`. El filtro con máscara booleana se queda con los costos mayores que cero.

In [ ]:
costos = np.array([doc["costo_total"] for doc in intervenciones], dtype=float)
print("media:", costos.mean(), "· mediana:", np.median(costos))
print("máximo:", costos.max(), "· mínimo:", costos.min())
con_costo = costos[costos > 0]
print("intervenciones con costo > 0:", con_costo.size, "→", con_costo)

## 3. De documentos a DataFrame con `pd.json_normalize`

`json_normalize` aplana los subdocumentos: `camposDinamicos.spo2`, `equipo_snapshot.ubicacion`, etc. Los arreglos
(`repuestos`) se quedan como listas dentro de una celda. Se descartan las columnas «vacías» que aparecen cuando el
subdocumento es `null` (intervenciones sin firmar) o `{}` (tenant B, sin campos dinámicos).

In [ ]:
crudo = pd.json_normalize(intervenciones)
crudo = crudo.drop(columns=[c for c in ("equipo_snapshot", "camposDinamicos") if c in crudo.columns])
print(crudo.shape)
crudo.columns.tolist()

Una **Serie** de costos por equipo (índice = `equipo_id`):

In [ ]:
costo_por_equipo = crudo.groupby("equipo_id")["costo_total"].sum().sort_values(ascending=False)
costo_por_equipo

## 4. Inspección

In [ ]:
crudo[["tenant_id", "tipo_mantenimiento", "fecha_intervencion", "costo_total", "cerrada_firmada"]].head(3)

In [ ]:
crudo[["tenant_id", "tipo_mantenimiento", "fecha_intervencion", "costo_total", "cerrada_firmada"]].tail(3)

In [ ]:
crudo.dtypes.value_counts()

In [ ]:
crudo.info(memory_usage=False, show_counts=True)

In [ ]:
crudo.describe(include="all").T[["count", "unique", "top", "mean", "min", "max"]].head(12)

**Hallazgos de la inspección**

- 12 filas, una por intervención; `_id` coincide con `id_intervencion`.
- `equipo_snapshot.*` y `estado_equipo_salida` están vacíos en las **4 intervenciones sin firmar**: el snapshot se
  captura al firmar (decisión del modelo documental de L1). No es un error de datos.
- `camposDinamicos.*` solo existe en el tenant A (monitores y bombas); en el tenant B son nulos porque no tiene
  configuración de campos. Cualquier promedio de esos campos debe declarar su universo.
- `fecha_intervencion` llega como fecha sin zona: se interpreta como **UTC**, que es como la guarda MongoDB.
- Columnas útiles para HU-008: `tenant_id`, `equipo_id`, `tipo_mantenimiento`, `fecha_intervencion`, `costo_total`,
  `tiempo_atencion_min`, `cerrada_firmada`; para HU-009 se suman `camposDinamicos.*` por tipo de equipo.

## 5. Copia de trabajo y bitácora de reglas (semana 5)

Desde aquí se trabaja sobre `df = crudo.copy()`: la fuente en disco y el DataFrame `crudo` no se tocan. Cada regla
que marca, calcula o normaliza se anota en la **bitácora** con su identificador, la columna que produce y cuántas
filas afectó. Así cada valor del CSV derivado se puede explicar.

In [ ]:
df = crudo.copy()
bitacora: list[dict] = []


def registrar(regla: str, descripcion: str, columna: str, filas: int) -> None:
    bitacora.append(
        {
            "regla": regla,
            "descripcion": descripcion,
            "columna": columna,
            "filas_afectadas": int(filas),
            "huella_fuente": HUELLA_FUENTE,
        }
    )


df["fecha_intervencion"] = pd.to_datetime(df["fecha_intervencion"], utc=True)
registrar("R01", "fecha_intervencion se interpreta en UTC", "fecha_intervencion", len(df))
df[["fecha_intervencion"]].head(2)

### 5.1 Fecha de corte (marcar, no borrar)

El lote analítico se corta el **21/10/2026 00:00 UTC** (inicio incluido, fin excluido: todo lo anterior entra en el
periodo). Las intervenciones posteriores **no se eliminan**: se marcan con `posterior_corte` para que los indicadores
decidan si las incluyen y lo digan.

In [ ]:
FECHA_CORTE = pd.Timestamp("2026-10-21T00:00:00Z")
df["posterior_corte"] = df["fecha_intervencion"] >= FECHA_CORTE
registrar(
    "R02",
    f"posterior a la fecha de corte {FECHA_CORTE.isoformat()}",
    "posterior_corte",
    df["posterior_corte"].sum(),
)
df.loc[df["posterior_corte"], ["tenant_id", "tipo_mantenimiento", "fecha_intervencion"]]

### 5.2 Normalizar texto: `replace()` frente a `map()`

Ambos traducen valores con un diccionario, pero se comportan distinto con lo que **no** está en el diccionario:

- `replace()` deja el valor original. Es seguro para corregir unas pocas variantes, pero una categoría nueva pasa
  sin que nadie lo note.
- `map()` devuelve `NaN`. Obliga a que el diccionario esté completo y permite **detectar** categorías inesperadas.

Se prueba con un diccionario incompleto (falta `calibracion`) para ver la diferencia.

In [ ]:
incompleto = {"preventivo": "Preventivo", "correctivo": "Correctivo"}
pd.DataFrame(
    {
        "original": df["tipo_mantenimiento"],
        "replace": df["tipo_mantenimiento"].replace(incompleto),
        "map": df["tipo_mantenimiento"].map(incompleto),
    }
).drop_duplicates()

Con el diccionario completo se usa `map()` y un `assert` garantiza que no quedó ninguna categoría sin traducir.

In [ ]:
ETIQUETAS_TIPO = {"preventivo": "Preventivo", "correctivo": "Correctivo", "calibracion": "Calibración"}
df["tipo_etiqueta"] = df["tipo_mantenimiento"].map(ETIQUETAS_TIPO)
assert df["tipo_etiqueta"].notna().all(), "Hay tipos de mantenimiento sin etiqueta"
registrar("R03", "etiqueta legible del tipo de mantenimiento (map)", "tipo_etiqueta", len(df))

df["tenant"] = df["tenant_id"].map(TENANTS)
assert df["tenant"].notna().all(), "Hay intervenciones de un tenant fuera del laboratorio"
registrar("R04", "nombre del tenant de laboratorio (map)", "tenant", len(df))
df[["tipo_mantenimiento", "tipo_etiqueta", "tenant"]].drop_duplicates()

### 5.3 Columnas calculadas vectorizadas (sin inventar datos)

- `mes`: año y mes de la intervención en UTC.
- `firmada`: «Sí»/«No» para lectura humana.
- `num_repuestos` y `suma_repuestos`: los repuestos se «explotan» (una fila por repuesto) y se agregan por
  intervención. Una lista vacía significa *cero repuestos* (es un dato conocido), por eso ahí sí se pone 0.
- `costo_por_hora`: `costo_total / horas`. Si el tiempo no es mayor que cero el resultado queda vacío (`NaN`), no se
  rellena con un número inventado.

In [ ]:
df["mes"] = df["fecha_intervencion"].dt.strftime("%Y-%m")
df["firmada"] = np.where(df["cerrada_firmada"], "Sí", "No")
df["num_repuestos"] = df["repuestos"].str.len()

repuestos = df[["_id", "repuestos"]].explode("repuestos").dropna(subset=["repuestos"])
detalle = pd.json_normalize(repuestos["repuestos"].tolist()).assign(_id=repuestos["_id"].to_numpy())
subtotal = (detalle["cantidad"] * detalle["costo_unitario"]).groupby(detalle["_id"]).sum()
df["suma_repuestos"] = df["_id"].map(subtotal).fillna(0.0)

horas = df["tiempo_atencion_min"].where(df["tiempo_atencion_min"] > 0) / 60
df["costo_por_hora"] = (df["costo_total"] / horas).round(0)

for regla, columna in [
    ("R05", "mes"),
    ("R06", "firmada"),
    ("R07", "num_repuestos"),
    ("R08", "suma_repuestos"),
    ("R09", "costo_por_hora"),
]:
    registrar(regla, "columna calculada vectorizada", columna, df[columna].notna().sum())
df[["mes", "firmada", "num_repuestos", "suma_repuestos", "costo_total", "costo_por_hora"]].head(5)

### 5.4 Marcar conflictos (sin borrar ni corregir)

| Regla | Conflicto | Por qué importa |
|---|---|---|
| C1 | La ubicación del snapshot es distinta de la ubicación actual del equipo (PostgreSQL) | Es el «cambio histórico» documentado: los reportes usan el snapshot y la hoja de vida el dato actual. Solo se evalúa si hay snapshot; si no lo hay queda `<NA>` (no evaluable), no `False`. |
| C2 | Intervención **sin firmar** con `costo_total > 0` | Es un costo todavía no confirmado: no debe sumarse a los costos firmados. |
| C3 | `costo_total` distinto de la suma de los repuestos | Señala un costo mal calculado. |

Las ubicaciones se comparan normalizadas (`strip` + `casefold`) para no marcar como conflicto una diferencia de
mayúsculas o espacios.

In [ ]:
referencias = json_util.loads(REFERENCIAS.read_text(encoding="utf-8"))
ubicacion_actual = {
    equipo["id"]: equipo["ubicacion_actual"]
    for tenant in referencias["tenants"].values()
    for equipo in tenant["equipos"]
}
df["ubicacion_actual"] = df["equipo_id"].map(ubicacion_actual)


def normalizar(texto: pd.Series) -> pd.Series:
    return texto.astype("string").str.strip().str.casefold()


distinta = normalizar(df["equipo_snapshot.ubicacion"]) != normalizar(df["ubicacion_actual"])
df["conflicto_ubicacion"] = distinta.astype("boolean").mask(df["equipo_snapshot.ubicacion"].isna())
df["conflicto_costo_sin_firma"] = ~df["cerrada_firmada"] & (df["costo_total"] > 0)
df["conflicto_suma_repuestos"] = ~np.isclose(df["costo_total"], df["suma_repuestos"])
df["con_conflicto"] = (
    df["conflicto_ubicacion"].fillna(False) | df["conflicto_costo_sin_firma"] | df["conflicto_suma_repuestos"]
)

registrar(
    "C1",
    "ubicación del snapshot ≠ ubicación actual (solo con snapshot)",
    "conflicto_ubicacion",
    df["conflicto_ubicacion"].sum(),
)
registrar(
    "C2", "sin firmar con costo_total > 0", "conflicto_costo_sin_firma", df["conflicto_costo_sin_firma"].sum()
)
registrar(
    "C3", "costo_total ≠ suma de repuestos", "conflicto_suma_repuestos", df["conflicto_suma_repuestos"].sum()
)
registrar(
    "C0",
    "tiene al menos un conflicto (NA de C1 cuenta como sin conflicto)",
    "con_conflicto",
    df["con_conflicto"].sum(),
)

df[["conflicto_ubicacion", "conflicto_costo_sin_firma", "conflicto_suma_repuestos", "con_conflicto"]].agg(
    ["sum", "count"]
)

### 5.5 Prioridad de revisión con `np.where` y `np.select`

`np.where(condición, a, b)` resuelve una decisión **binaria**. `np.select(condiciones, valores, default)` resuelve
varias, y **gana la primera condición verdadera** en el orden de la lista. Por eso el orden es la regla de negocio:

| Orden | Prioridad | Condición |
|---|---|---|
| 1 | Alta | Tiene al menos un conflicto (C1, C2 o C3) |
| 2 | Media | Correctivo sin firmar, **o** posterior a la fecha de corte, **o** firmada con el equipo no operativo a la salida |
| — | Baja | Ninguna de las anteriores (`default`) |

Una correctiva sin firmar con costo cumple 1 y 2: queda en **Alta** porque la condición 1 va primero.

In [ ]:
df["estado_firma"] = np.where(df["cerrada_firmada"], "Cerrada", "Pendiente de firma")

no_operativo = df["cerrada_firmada"] & df["estado_equipo_salida"].ne("Operativo")
condiciones = [
    df["con_conflicto"],
    (df["tipo_mantenimiento"].eq("correctivo") & ~df["cerrada_firmada"])
    | df["posterior_corte"]
    | no_operativo,
]
df["prioridad_revision"] = np.select(condiciones, ["Alta", "Media"], default="Baja")

registrar("P01", "estado de firma (np.where)", "estado_firma", len(df))
registrar(
    "P02",
    "prioridad: 1) conflicto→Alta 2) correctivo sin firma/posterior al corte/no operativo→Media 3) Baja",
    "prioridad_revision",
    (df["prioridad_revision"] != "Baja").sum(),
)
df["prioridad_revision"].value_counts()

### 5.6 `apply` solo para explicar unas pocas filas

`apply(axis=1)` ejecuta una función de Python **fila por fila**: es lento y no aprovecha la vectorización. Aquí se
usa a propósito solo sobre las filas de prioridad Alta (pocas) para redactar una explicación legible; los cálculos
del conjunto completo se hicieron antes con operaciones vectorizadas.

In [ ]:
def explicar(fila: pd.Series) -> str:
    motivos = []
    if pd.notna(fila["conflicto_ubicacion"]) and fila["conflicto_ubicacion"]:
        antes, hoy = fila["equipo_snapshot.ubicacion"], fila["ubicacion_actual"]
        motivos.append(f"snapshot en «{antes}», hoy en «{hoy}»")
    if fila["conflicto_costo_sin_firma"]:
        motivos.append(f"costo de ${fila['costo_total']:,.0f} sin firma")
    if fila["conflicto_suma_repuestos"]:
        motivos.append("costo distinto de la suma de repuestos")
    return "; ".join(motivos)


alta = df.loc[df["prioridad_revision"] == "Alta"]
alta.apply(explicar, axis=1).set_axis(alta["_id"]).to_frame("motivo")

## 6. Validaciones con `assert`

Si alguna supuesta verdad sobre los datos deja de cumplirse, el notebook se detiene aquí en lugar de exportar
resultados equivocados.

In [ ]:
assert len(df) == len(crudo) == 12, "Se perdieron o duplicaron filas"
assert df["_id"].is_unique, "_id repetido"
assert (df["_id"] == df["id_intervencion"]).all(), "_id distinto de id_intervencion"
assert set(df["tenant_id"]) <= set(TENANTS), "tenant fuera del laboratorio"
assert (df["costo_total"] >= 0).all(), "costo negativo"
assert not df["conflicto_suma_repuestos"].any(), "costo_total no cuadra con los repuestos"
assert df.loc[df["cerrada_firmada"], "equipo_snapshot.nombre"].notna().all(), "firmada sin snapshot"
assert df["prioridad_revision"].isin(["Alta", "Media", "Baja"]).all()
assert sha256(FUENTE) == HUELLA_FUENTE, "La fuente cambió durante el proceso"
print("Validaciones superadas.")

## 7. Indicadores con universo explícito

Un porcentaje sin denominador no se puede interpretar. Cada indicador declara su **universo** (qué filas cuentan
en el denominador), el numerador y el denominador.

In [ ]:
def indicador(
    nombre: str,
    universo: str,
    en_universo: pd.Series,
    valor: pd.Series | None = None,
    cumple: pd.Series | None = None,
) -> dict:
    """Proporción (si se da `cumple`) o promedio (si se da `valor`) sobre las filas del universo."""
    base = df.loc[en_universo]
    if cumple is not None:
        numerador = int(cumple[en_universo].sum())
        resultado = numerador / len(base) if len(base) else np.nan
    else:
        datos = valor[en_universo].dropna()
        numerador, resultado = float(datos.sum()), (datos.mean() if len(datos) else np.nan)
        base = datos
    return {
        "indicador": nombre,
        "universo": universo,
        "numerador": numerador,
        "denominador": len(base),
        "valor": round(float(resultado), 4),
    }


todas = pd.Series(True, index=df.index)
antes_corte = ~df["posterior_corte"]
correctivo_firmado = df["tipo_mantenimiento"].eq("correctivo") & df["cerrada_firmada"]
filas = [
    indicador(
        "Proporción de intervenciones firmadas",
        "todas las intervenciones",
        todas,
        cumple=df["cerrada_firmada"],
    ),
    indicador(
        "Proporción de intervenciones firmadas",
        "intervenciones anteriores a la fecha de corte",
        antes_corte,
        cumple=df["cerrada_firmada"],
    ),
    indicador(
        "Proporción con prioridad Alta",
        "todas las intervenciones",
        todas,
        cumple=df["prioridad_revision"].eq("Alta"),
    ),
    indicador(
        "Costo promedio (COP)",
        "correctivos firmados (costo confirmado)",
        correctivo_firmado,
        valor=df["costo_total"],
    ),
    indicador(
        "Tiempo de atención promedio (min)",
        "intervenciones anteriores a la fecha de corte",
        antes_corte,
        valor=df["tiempo_atencion_min"],
    ),
    indicador(
        "SpO2 promedio en la prueba (%)",
        "intervenciones con el campo dinámico spo2 registrado",
        df["camposDinamicos.spo2"].notna(),
        valor=df["camposDinamicos.spo2"],
    ),
]
for tenant_id, nombre in TENANTS.items():
    del_tenant = df["tenant_id"].eq(tenant_id) & antes_corte
    filas.append(
        indicador(
            "Proporción de intervenciones firmadas",
            f"{nombre}, anteriores a la fecha de corte",
            del_tenant,
            cumple=df["cerrada_firmada"],
        )
    )
indicadores = pd.DataFrame(filas)
indicadores

Agrupación por tenant y tipo de mantenimiento (todas las intervenciones):

In [ ]:
resumen = df.groupby(["tenant", "tipo_etiqueta"]).agg(
    intervenciones=("_id", "size"),
    firmadas=("cerrada_firmada", "sum"),
    costo_total=("costo_total", "sum"),
    tiempo_medio_min=("tiempo_atencion_min", "mean"),
)
resumen

## 8. Exportar y verificar

Se exportan tres archivos a `analytics/data/processed/` y un libro de Excel con indicadores y bitácora:

| Archivo | Contenido |
|---|---|
| `intervenciones_lab.csv` | Conjunto derivado: una fila por intervención, sin listas ni textos libres, con marcas y prioridad. |
| `indicadores_lab.csv` | Indicadores con universo, numerador y denominador. |
| `bitacora_reglas_lab.csv` | Reglas aplicadas, columnas producidas, filas afectadas y huella de la fuente. |
| `resumen_lab.xlsx` | Hojas `indicadores` y `bitacora` (openpyxl). |

Después se **releen** los CSV para comprobar que lo escrito es lo que se esperaba.

In [ ]:
COLUMNAS_SALIDA = [
    "_id",
    "tenant_id",
    "tenant",
    "equipo_id",
    "equipo_snapshot.tipo_equipo",
    "tipo_mantenimiento",
    "tipo_etiqueta",
    "fecha_intervencion",
    "mes",
    "posterior_corte",
    "costo_total",
    "suma_repuestos",
    "num_repuestos",
    "tiempo_atencion_min",
    "costo_por_hora",
    "cerrada_firmada",
    "firmada",
    "estado_firma",
    "estado_equipo_salida",
    "equipo_snapshot.ubicacion",
    "ubicacion_actual",
    "conflicto_ubicacion",
    "conflicto_costo_sin_firma",
    "conflicto_suma_repuestos",
    "con_conflicto",
    "prioridad_revision",
    "camposDinamicos.spo2",
    "camposDinamicos.frecuencia_cardiaca",
]
derivado = df[COLUMNAS_SALIDA].assign(
    fecha_intervencion=df["fecha_intervencion"].dt.strftime("%Y-%m-%dT%H:%M:%SZ")
)
rutas = {
    "derivado": SALIDA / "intervenciones_lab.csv",
    "indicadores": SALIDA / "indicadores_lab.csv",
    "bitacora": SALIDA / "bitacora_reglas_lab.csv",
}
derivado.to_csv(rutas["derivado"], index=False, encoding="utf-8", lineterminator="\n")
registrar("X01", "exportación del conjunto derivado", "intervenciones_lab.csv", len(derivado))
bitacora_df = pd.DataFrame(bitacora)
indicadores.to_csv(rutas["indicadores"], index=False, encoding="utf-8", lineterminator="\n")
bitacora_df.to_csv(rutas["bitacora"], index=False, encoding="utf-8", lineterminator="\n")

with pd.ExcelWriter(SALIDA / "resumen_lab.xlsx", engine="openpyxl") as libro:
    indicadores.to_excel(libro, sheet_name="indicadores", index=False)
    bitacora_df.to_excel(libro, sheet_name="bitacora", index=False)

for nombre, ruta in rutas.items():
    print(f"{nombre:12} {ruta.relative_to(RAIZ).as_posix():50} sha256={sha256(ruta)[:16]}…")

In [ ]:
releido = pd.read_csv(rutas["derivado"], encoding="utf-8")
assert len(releido) == len(df) and releido.columns.tolist() == COLUMNAS_SALIDA
assert releido["_id"].tolist() == df["_id"].tolist()
assert releido["prioridad_revision"].value_counts().equals(df["prioridad_revision"].value_counts())

releidos_ind = pd.read_csv(rutas["indicadores"], encoding="utf-8")
assert len(releidos_ind) == len(indicadores)
assert np.allclose(releidos_ind["valor"], indicadores["valor"], equal_nan=True)

releida_bit = pd.read_csv(rutas["bitacora"], encoding="utf-8")
assert (releida_bit["huella_fuente"] == HUELLA_FUENTE).all()
assert pd.read_excel(SALIDA / "resumen_lab.xlsx", sheet_name="bitacora").shape == bitacora_df.shape

assert sha256(FUENTE) == HUELLA_FUENTE, "La fuente cambió"
print("Archivos releídos y verificados; la fuente sigue intacta:", HUELLA_FUENTE[:16], "…")
releida_bit[["regla", "columna", "filas_afectadas"]]

## 9. Conclusiones

- La semilla tiene 12 intervenciones coherentes: ningún costo descuadra con sus repuestos (C3 = 0) y toda
  intervención firmada tiene snapshot.
- Los conflictos encontrados son **marcas para revisar**, no errores que se corrijan aquí: el cambio de ubicación
  del monitor 01 (C1, esperado por diseño) y un costo registrado en una correctiva sin firmar (C2).
- La fecha de corte separa 4 intervenciones de finales de octubre; siguen en el conjunto derivado, marcadas.
- Los indicadores dependen del universo: la proporción de firmadas cambia si se cuentan o no las posteriores al corte,
  y el SpO2 promedio solo representa a los monitores del tenant A.
- La huella SHA-256 en la bitácora une cada archivo exportado con la versión exacta de la fuente. Este conjunto
  derivado es la entrada prevista para los indicadores de HU-008 y la agrupación de equipos de HU-009.